# 03 — Training: Per-Daerah Isolation Forest + Master Bundle

**Goal:** issolation forest per daeraeh untuk reasonign menggunakan SHAP, and package everything into a single
`master_model.joblib`.

**Pipeline at a glance:**
```
cleaned_data.csv
    -> split by daerah
        -> feature engineering
            -> time-based train/test split (no leakage)
                -> IsolationForest fit + SHAP explain
                    -> per-daerah artifacts
                        -> master_model.joblib + summary CSVs
```



## 1 · Imports & Configuration

In [ ]:
import json
import re
import shutil
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ── Paths ──────────────────────────────────────────────────────────────────────
CLEANED_DATA_PATH = Path("../data/cleaned/cleaned_data.csv")
TRAIN_DIR         = Path("../data/cleaned/train")
TEST_DIR          = Path("../data/cleaned/test")
ARTIFACT_ROOT     = Path("../artifacts/post_award_anomaly")
REGION_ROOT       = ARTIFACT_ROOT / "by_daerah"
MASTER_MODEL_PATH = ARTIFACT_ROOT / "master_model.joblib"
MASTER_MANIFEST   = ARTIFACT_ROOT / "master_model_manifest.json"

# ── Hyper-parameters ───────────────────────────────────────────────────────────
TRAIN_RATIO         = 0.85   # target fraction of rows used for training
CONTAMINATION       = 0.03   # expected anomaly rate (3%)
RANDOM_STATE        = 42
MIN_ROWS_PER_REGION = 100    # skip regions with fewer rows ( TIDAK VALID DATA KECIL ) RESOLVE NAN

# ── Feature schema ─────────────────────────────────────────────────────────────
NUMERIC_FEATURES = [
    "tender_minvalue", "award_value", "days_to_award",
    "budget_utilization_ratio", "value_gap",
    "log_tender_minvalue", "log_award_value",
    "title_word_count", "award_title_word_count",
    "supplier_count", "award_value_per_day",
    "same_day_award_flag", "award_month", "award_quarter", "award_weekday",
]
CATEGORICAL_FEATURES = ["mainprocurementcategory"]
FEATURE_COLUMNS      = NUMERIC_FEATURES + CATEGORICAL_FEATURES

BASE_EXPORT_COLUMNS = [
    "lspe_id", "nama_daerah", "date", "buyer_name", "tender_title",
    "mainprocurementcategory", "tender_minvalue", "tender_status",
    "award_title", "award_date", "award_value", "award_supplier",
    "days_to_award", "budget_utilization_ratio",
]

# ── Reset output directories so every run starts clean ────────────────────────
for _dir in [TRAIN_DIR, TEST_DIR, ARTIFACT_ROOT]:
    if _dir.exists():
        for item in _dir.iterdir():
            shutil.rmtree(item) if item.is_dir() else item.unlink()
    _dir.mkdir(parents=True, exist_ok=True)

REGION_ROOT.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Train ratio         : {TRAIN_RATIO}")
print(f"  Contamination       : {CONTAMINATION}")
print(f"  Min rows per daerah : {MIN_ROWS_PER_REGION}")
print(f"  SHAP version        : {shap.__version__}")
print("  Output folders      : reset")


Configuration loaded.
  Train ratio         : 0.85
  Contamination       : 0.03
  Min rows per daerah : 100
  SHAP version        : 0.51.0
  Output folders      : reset


## 2 · Helper Functions

All reusable logic lives here so the training loop stays readable.
Each function has a docstring explaining *what* and *why*.

In [ ]:
# ── Utility helpers ────────────────────────────────────────────────────────────

def slug(text: str) -> str:
    """Convert any string to a filesystem-safe slug (e.g. 'Jawa Tengah' -> 'jawa_tengah')."""
    return re.sub(r"[^a-z0-9]+", "_", str(text).strip().lower()).strip("_")

# numpy pandas to python syntax data type
def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.integer):  return int(value)
    if isinstance(value, np.floating): return float(value)
    if isinstance(value, np.bool_):    return bool(value)
    if isinstance(value, pd.Timestamp): return value.isoformat()
    return value

# SHAP explanation to human readable
def format_number(value) -> str:
    if pd.isna(value): return "missing"
    if isinstance(value, (int, np.integer)):     return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):  return f"{value:,.4f}".rstrip("0").rstrip(".")
    return str(value)


# ── Feature engineering ────────────────────────────────────────────────────────
# generalization and normalization before model
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data["date"]       = pd.to_datetime(data["date"],       errors="coerce")
    data["award_date"] = pd.to_datetime(data["award_date"], errors="coerce")

    # Calendar features from the award date
    data["award_month"]   = data["award_date"].dt.month
    data["award_quarter"] = data["award_date"].dt.quarter
    data["award_weekday"] = data["award_date"].dt.weekday

    # Log-transform to reduce skew on monetary values
    data["log_tender_minvalue"] = np.log1p(data["tender_minvalue"].clip(lower=0))
    data["log_award_value"]     = np.log1p(data["award_value"].clip(lower=0))

    # Derived monetary signal
    data["value_gap"] = data["award_value"] - data["tender_minvalue"]

    # Text-length features
    data["title_word_count"]       = data["tender_title"].fillna("").astype(str).str.split().str.len()
    data["award_title_word_count"] = data["award_title"].fillna("").astype(str).str.split().str.len()

    # Supplier participation
    tokens = data["award_supplier"].fillna("").astype(str).str.split(",")
    data["supplier_count"] = tokens.apply(
        lambda items: max(len([i for i in items if str(i).strip()]), 1)
    )

    # Speed of spend — guard against zero days_to_award
    safe_days = data["days_to_award"].replace(0, 1)
    data["award_value_per_day"] = data["award_value"] / safe_days
    data["same_day_award_flag"] = (data["days_to_award"] == 0).astype(int)

    return data


# ── Preprocessing pipeline ─────────────────────────────────────────────────────

# Column Transformer numeric vs categorical
def make_preprocessor() -> ColumnTransformer:
    try:
        # OneHotEncoder text to binary
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    # Numeric pipeline: median imputation + standard scaling
    # simpleimputer to handle missing values by replacing them with the median of each column
    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                         ("scaler",  StandardScaler())])
    # Categorical pipeline: most-frequent imputation + one-hot encoding
    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                         ("encoder", encoder)])

    return ColumnTransformer([
        ("num", num_pipe, NUMERIC_FEATURES),
        ("cat", cat_pipe, CATEGORICAL_FEATURES),
    ])


# Strip sklearn ColumnTransformer renaming.
def clean_feature_names(raw_names) -> list:
    cleaned = []
    for name in raw_names:
        if name.startswith("num__"):
            cleaned.append(name.replace("num__", ""))
        elif name.startswith("cat__mainprocurementcategory_"):
            cleaned.append("cat_" + name.replace("cat__mainprocurementcategory_", ""))
        else:
            cleaned.append(name)
    return cleaned


# ── SHAP aggregation ───────────────────────────────────────────────────────────

# When using one-hot encoding, SHAP values are generated for each dummy variable rather than the original categorical feature (mainprocurementcategory).
# This function sums the SHAP values of the dummy variables back to their parent feature for easier interpretation.
def aggregate_shap_by_raw_feature(row_shap, feature_names: list) -> dict:
    aggregated = {}
    for feat, val in zip(feature_names, row_shap):
        raw = "mainprocurementcategory" if feat.startswith("cat_") else feat
        aggregated[raw] = aggregated.get(raw, 0.0) + float(val)
    return aggregated


# ── Scoring helpers ────────────────────────────────────────────────────────────

# anomaly : low mediunm high by certain theshold
def assign_severity(scores, medium_cutoff: float, anomaly_threshold: float):
    return np.select(
        [scores >= anomaly_threshold, scores >= medium_cutoff],
        ["high", "medium"],
        default="low",
    )

# Attach anomaly scores and labels to a DataFrame.
def make_scored_output(df: pd.DataFrame, scores,
    medium_cutoff: float, anomaly_threshold: float) -> pd.DataFrame:
    out = df.reset_index(drop=True).copy()
    out["anomaly_score"]    = scores
    out["severity_band"]    = assign_severity(scores, medium_cutoff, anomaly_threshold)
    out["prediction_label"] = np.select(
        [scores >= anomaly_threshold, scores >= medium_cutoff],
        ["anomaly", "warning"],
        default="normal",
    )
    out["anomaly_flag"] = (scores >= anomaly_threshold).astype(int)
    return out


# Add top 3 SHAP feature columns + human-readable explanation to every row.
def build_local_explanations(df: pd.DataFrame, shap_values, feature_names: list) -> pd.DataFrame:
    out = df.reset_index(drop=True).copy()
    for rank in [1, 2, 3]:
        out[f"top_{rank}_feature"] = None
        out[f"top_{rank}_shap"]    = float("nan")
        out[f"top_{rank}_impact"]  = None

    explanations = []
    for idx in range(len(out)):
        aggregated   = aggregate_shap_by_raw_feature(shap_values[idx], feature_names)
        top_features = sorted(aggregated.items(), key=lambda x: abs(x[1]), reverse=True)[:3]

        parts = []
        for rank, (feat, shap_val) in enumerate(top_features, start=1):
            raw_val = out.loc[idx, feat] if feat in out.columns else out.loc[idx, "mainprocurementcategory"]
            impact  = "pushes anomaly score up" if shap_val > 0 else "pushes anomaly score down"
            out.loc[idx, f"top_{rank}_feature"] = feat
            out.loc[idx, f"top_{rank}_shap"]    = float(shap_val)
            out.loc[idx, f"top_{rank}_impact"]  = impact
            parts.append(f"{feat}={format_number(raw_val)} ({impact}, SHAP {shap_val:.4f})")

        severity = out.loc[idx, "severity_band"].upper()
        explanations.append(f"[{severity}] " + "; ".join(parts))

    out["human_readable_explanation"] = explanations
    return out


# ── Reference thresholds ───────────────────────────────────────────────────────

# Compute per-category statistical thresholds from the training set.
# Stored alongside the model so inference can run rule-based checks too.
# Returns (thresholds_dict, category_reference_df).
def build_reference_thresholds(train_output: pd.DataFrame):
    category_ref = (
        train_output
        .groupby("mainprocurementcategory")
        .agg(
            # For each category, compute the 99th percentile of award_value, 95th percentile of days_to_award,
            award_value_p99   =("award_value",              lambda s: float(s.quantile(0.99))),
            days_to_award_p95 =("days_to_award",            lambda s: float(s.quantile(0.95))),
            # For budget utilization ratio, compute both the 5th and 95th percentiles to capture unusually low or high utilization
            budget_ratio_p05  =("budget_utilization_ratio", lambda s: float(s.quantile(0.05))),
            budget_ratio_p95  =("budget_utilization_ratio", lambda s: float(s.quantile(0.95))),
        )
        .reset_index()
    )
    # For the entire training set, compute the same thresholds to have a global reference point.
    global_ref = {
        "award_value_p99":   float(train_output["award_value"].quantile(0.99)),
        "days_to_award_p95": float(train_output["days_to_award"].quantile(0.95)),
        "budget_ratio_p05":  float(train_output["budget_utilization_ratio"].quantile(0.05)),
        "budget_ratio_p95":  float(train_output["budget_utilization_ratio"].quantile(0.95)),
    }
    # global thresholda
    thresholds = {
        "category_reference": category_ref.to_dict(orient="records"),
        "global_reference":   global_ref,
    }
    return thresholds, category_ref


# ── Demo case selection ────────────────────────────────────────────────────────

def select_demo_cases(test_output: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for label in ["normal", "warning", "anomaly"]:
        subset = test_output[test_output["prediction_label"] == label].copy()
        if subset.empty:
            continue
        if label == "anomaly":
            chosen = subset.sort_values("anomaly_score", ascending=False).head(1)
        else:
            subset["_dist"] = (subset["anomaly_score"] - subset["anomaly_score"].median()).abs()
            chosen = subset.sort_values("_dist").head(1).drop(columns=["_dist"])
        frames.append(chosen)

    return pd.concat(frames, ignore_index=True) if frames else test_output.head(0).copy()


# ── Time-based train / test split ─────────────────────────────────────────────

# split on award_date so train precedes test — no date overlap.
# return value error jika tidak ada cutoff date yang valid
def make_time_based_split(df: pd.DataFrame, train_ratio: float):
    date_counts = (
        df.groupby("award_date").size()
          .rename("row_count").reset_index()
          .sort_values("award_date").reset_index(drop=True)
    )
    if len(date_counts) < 2:
        raise ValueError("Not enough unique award_date values for a time split.")

    n = len(df)
    min_train = max(50,  min(300, int(n * 0.30)))
    min_test  = max(20,  min(100, int(n * 0.10)))

    candidates = []
    for cutoff in date_counts["award_date"].iloc[1:]:
        n_train = int((df["award_date"] < cutoff).sum())
        n_test  = int((df["award_date"] >= cutoff).sum())
        candidates.append({
            "cutoff_date":        cutoff,
            "train_rows":         n_train,
            "test_rows":          n_test,
            "train_ratio_actual": n_train / n,
        })

    candidates_df = pd.DataFrame(candidates)
    valid = candidates_df[
        (candidates_df["train_rows"] >= min_train) &
        (candidates_df["test_rows"]  >= min_test)
    ].copy()

    if valid.empty:
        raise ValueError(
            f"No valid cutoff (min_train={min_train}, min_test={min_test})."
        )

    valid["ratio_gap"] = (valid["train_ratio_actual"] - train_ratio).abs()
    best = valid.sort_values(["ratio_gap", "cutoff_date"]).iloc[0]
    cutoff = best["cutoff_date"]

    train_df = df[df["award_date"] <  cutoff].reset_index(drop=True).copy()
    test_df  = df[df["award_date"] >= cutoff].reset_index(drop=True).copy()

    split_info = {
        "cutoff_date":        cutoff,
        "train_rows":         int(best["train_rows"]),
        "test_rows":          int(best["test_rows"]),
        "train_ratio_actual": float(best["train_ratio_actual"]),
        "min_train_rows":     min_train,
        "min_test_rows":      min_test,
        "candidate_count":    int(len(valid)),
    }
    return train_df, test_df, split_info


print("Helper functions defined.")


Helper functions defined.


## 3 · Load Data

Read the master cleaned CSV and show a per-region summary before training.

In [ ]:
data = pd.read_csv(CLEANED_DATA_PATH)
data["lspe_id"]    = data["lspe_id"].astype(str)
data["date"]       = pd.to_datetime(data["date"],       errors="coerce")
data["award_date"] = pd.to_datetime(data["award_date"], errors="coerce")
data = data.sort_values(["nama_daerah", "award_date", "date"]).reset_index(drop=True)

# data regroup by time and region jadi setiap provinsi di pisah 
region_summary = (
    data.groupby(["nama_daerah", "lspe_id"])
        .agg(
            rows           =("nama_daerah", "size"),
            min_award_date =("award_date", "min"),
            max_award_date =("award_date", "max"),
        )
        .reset_index()
        .sort_values(["nama_daerah", "lspe_id"])
        .reset_index(drop=True)
)

print(f"Total rows : {len(data):,}")
print(f"Regions    : {len(region_summary)}")
display(region_summary)


Total rows : 145,921
Regions    : 34


,nama_daerah,lspe_id,rows,min_award_date,max_award_date
0,Aceh,106,12475,2015-03-09,2023-10-25
1,Bali,33,1934,2014-12-10,2023-10-10
2,Banten,99,3452,2015-01-13,2023-11-02
3,Bengkulu,267,1974,2015-02-15,2023-11-06
4,DKI Jakarta,127,8678,2014-12-24,2023-10-18
5,Daerah Istimewa Yogyakarta,13,4537,2015-01-23,2016-05-20
6,Gorontalo,18,1629,NaT,NaT
7,Jambi,70,3000,2015-12-21,2023-09-30
8,Jawa Barat,14,10629,2014-12-15,2022-07-22
9,Jawa Tengah,42,6038,2020-01-29,2023-10-16


## 4 · Train One Model per Daerah

For each region this loop:
1. Engineers features
2. Does a **time-based train/test split** — train always precedes test, no leakage
3. Fits an **IsolationForest** on the training set
4. Derives severity thresholds from the training score distribution
5. Runs **SHAP TreeExplainer** on the test set for interpretability
6. Saves all artifacts to `by_daerah/<region_slug>/`


In [4]:
# Accumulators — filled region by region, consumed in Cell 5
master_regions          = {}
master_summary_rows     = []
master_train_outputs    = []
master_test_outputs     = []
master_shap_outputs     = []
master_reference_outputs = []
master_demo_inputs      = []
master_demo_outputs     = []

# ── Main training loop ─────────────────────────────────────────────────────────
for region in region_summary.itertuples(index=False):
    nama_daerah = region.nama_daerah
    lspe_id     = str(region.lspe_id)
    total_rows  = int(region.rows)
    region_key  = nama_daerah.strip().casefold()
    region_slug = f"{slug(nama_daerah)}_{lspe_id}"
    region_dir  = REGION_ROOT / region_slug
    region_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 70)
    print(f"  DAERAH: {nama_daerah}  (id={lspe_id}, rows={total_rows:,})")
    print("=" * 70)

    # ── Guard: skip regions with insufficient data ─────────────────────────
    if total_rows < MIN_ROWS_PER_REGION:
        print(f"  Skipped: fewer than {MIN_ROWS_PER_REGION} rows.")
        master_summary_rows.append({
            "lspe_id": lspe_id, "nama_daerah": nama_daerah,
            "status": "skipped_not_enough_rows", "total_rows": total_rows,
            "artifact_subdir": str(region_dir.relative_to(ARTIFACT_ROOT)),
        })
        continue

    # ── Feature engineering ────────────────────────────────────────────────
    region_data = (
        data[data["nama_daerah"] == nama_daerah]
            .copy()
            .sort_values(["award_date", "date"])
            .reset_index(drop=True)
    )
    region_data = engineer_features(region_data)

    # ── Time-based train/test split ────────────────────────────────────────
    try:
        train_data, test_data, split_info = make_time_based_split(region_data, TRAIN_RATIO)
    except ValueError as exc:
        print(f"  Skipped: time split failed: {exc}")
        master_summary_rows.append({
            "lspe_id": lspe_id, "nama_daerah": nama_daerah,
            "status": "skipped_split_failed", "total_rows": total_rows,
            "artifact_subdir": str(region_dir.relative_to(ARTIFACT_ROOT)),
            "split_reason": str(exc),
        })
        continue

    train_max_date = train_data["award_date"].max()
    test_min_date  = test_data["award_date"].min()

    # Hard stop if the split is not temporally clean
    if train_max_date > test_min_date:
        raise ValueError(
            f"Temporal leakage in {nama_daerah}: "
            f"train ends {train_max_date}, test starts {test_min_date}"
        )

    print(f"  Train: {len(train_data):,} rows  (up to {train_max_date.date()})")
    print(f"  Test : {len(test_data):,} rows  (from {test_min_date.date()})")
    print(f"  Actual train ratio: {split_info['train_ratio_actual']:.3f}")

    # ── Preprocessing ──────────────────────────────────────────────────────
    preprocessor = make_preprocessor()
    X_train = preprocessor.fit_transform(train_data[FEATURE_COLUMNS])
    X_test  = preprocessor.transform(test_data[FEATURE_COLUMNS])

    # ── IsolationForest ────────────────────────────────────────────────────
    model = IsolationForest(
        n_estimators=400,
        contamination=CONTAMINATION,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train)

    # Negate score_samples: higher value = more anomalous (more intuitive)
    train_scores = -model.score_samples(X_train)
    test_scores  = -model.score_samples(X_test)

    # Thresholds derived from the training score distribution
    medium_cutoff     = float(np.quantile(train_scores, 0.70))
    anomaly_threshold = float(np.quantile(train_scores, 1 - CONTAMINATION))
    print(f"  Thresholds — medium: {medium_cutoff:.6f}, anomaly: {anomaly_threshold:.6f}")

    # ── Scoring ────────────────────────────────────────────────────────────
    train_output = make_scored_output(train_data, train_scores, medium_cutoff, anomaly_threshold)
    test_output  = make_scored_output(test_data,  test_scores,  medium_cutoff, anomaly_threshold)

    print("\n  Label distribution:")
    print("    Train:", train_output["prediction_label"].value_counts().to_dict())
    print("    Test :", test_output["prediction_label"].value_counts().to_dict())

    # ── SHAP explanations ──────────────────────────────────────────────────
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    if isinstance(shap_values, list):   # older SHAP versions return a list
        shap_values = shap_values[0]

    feature_names = clean_feature_names(preprocessor.get_feature_names_out())
    test_output   = build_local_explanations(test_output, shap_values, feature_names)

    # Aggregate SHAP to raw features for global importance ranking
    shap_importance = pd.DataFrame({
        "feature":       feature_names,
        "mean_abs_shap": abs(shap_values).mean(axis=0),
    })
    shap_importance["raw_feature"] = shap_importance["feature"].apply(
        lambda f: "mainprocurementcategory" if f.startswith("cat_") else f
    )
    shap_global = (
        shap_importance.groupby("raw_feature", as_index=False)["mean_abs_shap"].sum()
                       .sort_values("mean_abs_shap", ascending=False)
                       .reset_index(drop=True)
    )
    shap_global.insert(0, "lspe_id",    lspe_id)
    shap_global.insert(1, "nama_daerah", nama_daerah)

    print("\n  Top-5 SHAP features:")
    display(shap_global[["raw_feature", "mean_abs_shap"]].head(5))

    # ── Reference thresholds ───────────────────────────────────────────────
    reference_thresholds, category_ref = build_reference_thresholds(train_output)
    category_ref.insert(0, "lspe_id",    lspe_id)
    category_ref.insert(1, "nama_daerah", nama_daerah)

    # ── Demo cases ─────────────────────────────────────────────────────────
    demo_cases = select_demo_cases(test_output)
    if not demo_cases.empty:
        demo_cases["demo_case_label"] = [
            f"{region_slug}_{label}_demo"
            for label in demo_cases["prediction_label"].tolist()
        ]

    # ── Save per-region artifacts ──────────────────────────────────────────
    train_data[BASE_EXPORT_COLUMNS].to_csv(TRAIN_DIR / f"train_data_{lspe_id}.csv", index=False)
    test_data[BASE_EXPORT_COLUMNS].to_csv(TEST_DIR   / f"test_data_{lspe_id}.csv",  index=False)

    train_data.to_csv(  region_dir / "train_post_award_split.csv",               index=False)
    test_data.to_csv(   region_dir / "test_post_award_split.csv",                index=False)
    train_output.to_csv(region_dir / "train_predictions.csv",                    index=False)
    test_output.to_csv( region_dir / "test_predictions_with_explanations.csv",   index=False)
    shap_global.to_csv( region_dir / "shap_global_importance.csv",               index=False)
    category_ref.to_csv(region_dir / "reference_thresholds_by_category.csv",    index=False)

    if not demo_cases.empty:
        demo_cases[BASE_EXPORT_COLUMNS + ["demo_case_label"]].to_csv(
            region_dir / "demo_input_cases.csv",    index=False)
        demo_cases.to_csv(region_dir / "demo_expected_outputs.csv", index=False)

    joblib.dump(model,        region_dir / "isolation_forest.joblib")
    joblib.dump(preprocessor, region_dir / "preprocessor.joblib")
    joblib.dump(explainer,    region_dir / "shap_explainer.joblib")

    # Metadata JSONs
    model_config = {
        "model_type":           "IsolationForest",
        "routing_mode":         "per_nama_daerah",
        "routing_key":          region_key,
        "region_name":          nama_daerah,
        "lspe_id":              lspe_id,
        "numeric_features":     NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "feature_columns":      FEATURE_COLUMNS,
        "contamination":        CONTAMINATION,
        "medium_cutoff":        medium_cutoff,
        "anomaly_threshold":    anomaly_threshold,
        "train_rows":           len(train_output),
        "test_rows":            len(test_output),
        "train_end_date":       train_max_date,
        "test_start_date":      test_min_date,
    }
    explanation_meta = {
        "region_name":              nama_daerah,
        "lspe_id":                  lspe_id,
        "feature_names_preprocessed": list(feature_names),
        "categorical_features":     CATEGORICAL_FEATURES,
        "routing_note":             "Master bundle routes by nama_daerah (casefold match).",
        "shap_note":                "Positive SHAP raises anomaly score; negative lowers it.",
    }

    (region_dir / "model_config.json").write_text(
        json.dumps(to_builtin(model_config), indent=2), encoding="utf-8")
    (region_dir / "explanation_meta.json").write_text(
        json.dumps(to_builtin(explanation_meta), indent=2), encoding="utf-8")
    (region_dir / "reference_thresholds.json").write_text(
        json.dumps(to_builtin(reference_thresholds), indent=2), encoding="utf-8")

    # ── Accumulate for master bundle ───────────────────────────────────────
    master_regions[region_key] = {
        "lspe_id": lspe_id, "nama_daerah": nama_daerah, "routing_key": region_key,
        "model": model, "preprocessor": preprocessor, "explainer": explainer,
        "model_config": model_config, "explanation_meta": explanation_meta,
        "reference_thresholds": reference_thresholds,
        "artifact_dir": str(region_dir.resolve()),
    }
    master_summary_rows.append({
        "lspe_id": lspe_id, "nama_daerah": nama_daerah, "status": "trained",
        "artifact_subdir":    str(region_dir.relative_to(ARTIFACT_ROOT)),
        "total_rows":         total_rows,
        "train_rows":         len(train_output),
        "test_rows":          len(test_output),
        "train_ratio_actual": split_info["train_ratio_actual"],
        "cutoff_date":        split_info["cutoff_date"],
        "train_end_date":     train_max_date,
        "test_start_date":    test_min_date,
        "medium_cutoff":      medium_cutoff,
        "anomaly_threshold":  anomaly_threshold,
        "train_anomaly_rate": float(train_output["anomaly_flag"].mean()),
        "test_anomaly_rate":  float(test_output["anomaly_flag"].mean()),
    })
    master_train_outputs.append(train_output)
    master_test_outputs.append(test_output)
    master_shap_outputs.append(shap_global)
    master_reference_outputs.append(category_ref)
    if not demo_cases.empty:
        master_demo_inputs.append(demo_cases[BASE_EXPORT_COLUMNS + ["demo_case_label"]])
        master_demo_outputs.append(demo_cases)

    print(f"\n  Saved to: {region_dir.resolve()}\n")

print("=" * 70)
print(f"Training complete. Regions trained: {len(master_regions)}")


  DAERAH: Aceh  (id=106, rows=12,475)
  Train: 10,604 rows  (up to 2022-03-20)
  Test : 1,871 rows  (from 2022-03-21)
  Actual train ratio: 0.850
  Thresholds — medium: 0.460699, anomaly: 0.575355

  Label distribution:
    Train: {'normal': 7423, 'warning': 2862, 'anomaly': 319}
    Test : {'normal': 1349, 'warning': 504, 'anomaly': 18}

  Top-5 SHAP features:


,raw_feature,mean_abs_shap
0,mainprocurementcategory,0.559214
1,award_quarter,0.236310
2,award_month,0.199296
3,title_word_count,0.186590
4,award_title_word_count,0.172008



  Saved to: C:\Users\VICTUS\coding\collab\3\artifacts\post_award_anomaly\by_daerah\aceh_106

  DAERAH: Bali  (id=33, rows=1,934)
  Train: 1,643 rows  (up to 2021-06-25)
  Test : 291 rows  (from 2021-06-28)
  Actual train ratio: 0.850
  Thresholds — medium: 0.473384, anomaly: 0.588264

  Label distribution:
    Train: {'normal': 1150, 'warning': 443, 'anomaly': 50}
    Test : {'normal': 173, 'warning': 98, 'anomaly': 20}

  Top-5 SHAP features:


,raw_feature,mean_abs_shap
0,mainprocurementcategory,0.482408
1,value_gap,0.300552
2,tender_minvalue,0.221231
3,award_value,0.180011
4,award_value_per_day,0.179547



  Saved to: C:\Users\VICTUS\coding\collab\3\artifacts\post_award_anomaly\by_daerah\bali_33

  DAERAH: Banten  (id=99, rows=3,452)
  Train: 2,934 rows  (up to 2022-03-15)
  Test : 518 rows  (from 2022-03-17)
  Actual train ratio: 0.850
  Thresholds — medium: 0.463712, anomaly: 0.576923

  Label distribution:
    Train: {'normal': 2054, 'warning': 792, 'anomaly': 88}
    Test : {'normal': 409, 'warning': 100, 'anomaly': 9}

  Top-5 SHAP features:


...


  Saved to: C:\Users\VICTUS\coding\collab\3\artifacts\post_award_anomaly\by_daerah\sulawesi_utara_173

  DAERAH: Sumatera Barat  (id=16, rows=4,150)
  Skipped: time split failed: Not enough unique award_date values for a time split.
  DAERAH: Sumatera Selatan  (id=103, rows=5,930)
  Train: 4,750 rows  (up to 2023-09-08)
  Test : 103 rows  (from 2023-09-12)
  Actual train ratio: 0.801
  Thresholds — medium: 0.458612, anomaly: 0.569815

  Label distribution:
    Train: {'normal': 3325, 'warning': 1282, 'anomaly': 143}
    Test : {'normal': 100, 'warning': 3}

  Top-5 SHAP features:


,raw_feature,mean_abs_shap
0,mainprocurementcategory,0.429005
1,award_quarter,0.309283
2,days_to_award,0.156546
3,award_title_word_count,0.132260
4,title_word_count,0.121834



  Saved to: C:\Users\VICTUS\coding\collab\3\artifacts\post_award_anomaly\by_daerah\sumatera_selatan_103

  DAERAH: Sumatera Utara  (id=27, rows=5,782)
  Train: 906 rows  (up to 2016-06-17)
  Test : 123 rows  (from 2016-06-24)
  Actual train ratio: 0.157
  Thresholds — medium: 0.472948, anomaly: 0.588743

  Label distribution:
    Train: {'normal': 634, 'warning': 244, 'anomaly': 28}
    Test : {'normal': 59, 'warning': 52, 'anomaly': 12}

  Top-5 SHAP features:


,raw_feature,mean_abs_shap
0,value_gap,0.517774
1,mainprocurementcategory,0.462737
2,award_weekday,0.352586
3,tender_minvalue,0.338090
4,award_value,0.311034



  Saved to: C:\Users\VICTUS\coding\collab\3\artifacts\post_award_anomaly\by_daerah\sumatera_utara_27

Training complete. Regions trained: 27


## 5 · Assemble Master Bundle & Save Summary

Combine every regional model into one `master_model.joblib` and write aggregated CSVs + manifest.

In [5]:
if not master_regions:
    raise RuntimeError("No daerah model was trained. Check data and MIN_ROWS_PER_REGION.")

# ── Summary table ──────────────────────────────────────────────────────────────
master_summary = (
    pd.DataFrame(master_summary_rows)
      .sort_values(["status", "nama_daerah"])
      .reset_index(drop=True)
)
master_summary.to_csv(ARTIFACT_ROOT / "master_training_summary.csv", index=False)

# ── Combined output CSVs ───────────────────────────────────────────────────────
if master_train_outputs:
    pd.concat(master_train_outputs, ignore_index=True).to_csv(
        ARTIFACT_ROOT / "master_train_predictions.csv", index=False)

if master_test_outputs:
    master_test_df = pd.concat(master_test_outputs, ignore_index=True)
    master_test_df.to_csv(
        ARTIFACT_ROOT / "master_test_predictions_with_explanations.csv", index=False)
else:
    master_test_df = pd.DataFrame()

if master_shap_outputs:
    all_shap = pd.concat(master_shap_outputs, ignore_index=True)
    all_shap.to_csv(ARTIFACT_ROOT / "master_shap_global_importance.csv", index=False)

    shap_aggregated = (
        all_shap.groupby("raw_feature", as_index=False)["mean_abs_shap"].mean()
                .sort_values("mean_abs_shap", ascending=False)
                .reset_index(drop=True)
    )
    shap_aggregated.to_csv(
        ARTIFACT_ROOT / "master_shap_global_importance_aggregated.csv", index=False)
else:
    shap_aggregated = pd.DataFrame(columns=["raw_feature", "mean_abs_shap"])

if master_reference_outputs:
    pd.concat(master_reference_outputs, ignore_index=True).to_csv(
        ARTIFACT_ROOT / "master_reference_thresholds_by_category.csv", index=False)

if master_demo_inputs:
    pd.concat(master_demo_inputs,  ignore_index=True).to_csv(
        ARTIFACT_ROOT / "master_demo_input_cases.csv",     index=False)
    pd.concat(master_demo_outputs, ignore_index=True).to_csv(
        ARTIFACT_ROOT / "master_demo_expected_outputs.csv", index=False)

# ── Master bundle ──────────────────────────────────────────────────────────────
master_bundle = {
    "bundle_type":      "multi_daerah_isolation_forest_router",
    "routing_column":   "nama_daerah",
    "routing_strategy": "casefold_exact_match",
    "shared_feature_contract": {
        "numeric_features":     NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "base_export_columns":  BASE_EXPORT_COLUMNS,
    },
    "region_count":    len(master_regions),
    "regions":         master_regions,
    "training_summary": master_summary.to_dict(orient="records"),
}
joblib.dump(master_bundle, MASTER_MODEL_PATH)

# Lightweight manifest (no heavy model objects, safe to inspect as JSON)
master_manifest = {
    "bundle_type":      master_bundle["bundle_type"],
    "routing_column":   master_bundle["routing_column"],
    "routing_strategy": master_bundle["routing_strategy"],
    "artifact_root":    str(ARTIFACT_ROOT.resolve()),
    "region_count":     len(master_regions),
    "regions": [
        {
            "routing_key":       key,
            "nama_daerah":       b["nama_daerah"],
            "lspe_id":           b["lspe_id"],
            "artifact_dir":      b["artifact_dir"],
            "anomaly_threshold": b["model_config"]["anomaly_threshold"],
            "medium_cutoff":     b["model_config"]["medium_cutoff"],
        }
        for key, b in master_regions.items()
    ],
}
MASTER_MANIFEST.write_text(json.dumps(to_builtin(master_manifest), indent=2), encoding="utf-8")

# ── Final summary print ────────────────────────────────────────────────────────
print("MASTER TRAINING SUMMARY")
print("=" * 70)
display(master_summary)

if not master_test_df.empty:
    print("\nCombined test label distribution:")
    print(master_test_df["prediction_label"].value_counts().to_string())

print("\nTop SHAP features (averaged across daerah):")
display(shap_aggregated.head(10))

print("\nArtifacts written:")
print(f"  {MASTER_MODEL_PATH.resolve()}")
print(f"  {MASTER_MANIFEST.resolve()}")
print(f"  Per-region dirs in: {REGION_ROOT.resolve()}")


MASTER TRAINING SUMMARY


,lspe_id,nama_daerah,status,artifact_subdir,total_rows,train_rows,test_rows,train_ratio_actual,cutoff_date,train_end_date,test_start_date,medium_cutoff,anomaly_threshold,train_anomaly_rate,test_anomaly_rate,split_reason
0,18,Gorontalo,skipped_split_failed,by_daerah\gorontalo_18,1629,NaN,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,Not enough unique award_date values for a time...
1,97,Kalimantan Barat,skipped_split_failed,by_daerah\kalimantan_barat_97,3015,NaN,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,Not enough unique award_date values for a time...
2,181,Kalimantan Selatan,skipped_split_failed,by_daerah\kalimantan_selatan_181,2460,NaN,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,Not enough unique award_date values for a time...
3,12,Kalimantan Tengah,skipped_split_failed,by_daerah\kalimantan_tengah_12,4120,NaN,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,Not enough unique award_date values for a time...
4,86,Kepulauan Bangka Belitung,skipped_split_failed,by_daerah\kepulauan_bangka_belitung_86,1342,NaN,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,"No valid cutoff (min_train=300, min_test=100)."
5,22,Kepulauan Riau,skipped_split_failed,by_daerah\kepulauan_riau_22,2777,NaN,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,Not enough unique award_date values for a time...
6,16,Sumatera Barat,skipped_split_failed,by_daerah\sumatera_barat_16,4150,NaN,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,Not enough unique award_date values for a time...
7,106,Aceh,trained,by_daerah\aceh_106,12475,10604.0,1871.0,0.850020,2022-03-21,2022-03-20,2022-03-21,0.460699,0.575355,0.030083,0.009621,NaN
8,33,Bali,trained,by_daerah\bali_33,1934,1643.0,291.0,0.849535,2021-06-28,2021-06-25,2021-06-28,0.473384,0.588264,0.030432,0.068729,NaN
9,99,Banten,trained,by_daerah\banten_99,3452,2934.0,518.0,0.849942,2022-03-17,2022-03-15,2022-03-17,0.463712,0.576923,0.029993,0.017375,NaN



Combined test label distribution:
prediction_label
normal     7977
warning    3592
anomaly     488

Top SHAP features (averaged across daerah):


,raw_feature,mean_abs_shap
0,mainprocurementcategory,0.498186
1,value_gap,0.222859
2,award_quarter,0.209432
3,budget_utilization_ratio,0.177452
4,award_month,0.176899
5,days_to_award,0.173042
6,award_title_word_count,0.171367
7,tender_minvalue,0.170834
8,title_word_count,0.167244
9,award_value_per_day,0.161876



Artifacts written:
  C:\Users\VICTUS\coding\collab\3\artifacts\post_award_anomaly\master_model.joblib
  C:\Users\VICTUS\coding\collab\3\artifacts\post_award_anomaly\master_model_manifest.json
  Per-region dirs in: C:\Users\VICTUS\coding\collab\3\artifacts\post_award_anomaly\by_daerah
